#  Part 3: The Advanced AI-Dubbing & Post-Production Pipeline

##  Project Overview
This notebook implements an end-to-end automated pipeline to translate, dub, and subtitle video content. The goal of this phase is to transition from raw translated text to a professional "Cinematic Dub" that preserves the original actor's vocal identity while maintaining perfect temporal synchronization and studio-quality audio clarity.

---

##  Engineering Choices & Strategy
* **Zero-Shot Voice Cloning:** We utilized the **XTTS-v2** model. Unlike standard TTS, this allows us to replicate the specific timber and tone of the original speaker using only a few seconds of reference audio, creating a seamless transition between languages.
* **Deep-Learning Source Separation:** We integrated **Demucs** to isolate vocals. This was a critical architectural choice; by stripping background music *before* cloning, we ensured the AI model learned only the speaker's voice, preventing background artifacts from being "cloned" into the new dialogue.
* **Dynamic Time-Stretching (The "Gap Squisher"):** To solve the "expansion" problem (where French translations are often 20-30% longer than the original English), we engineered custom logic that calculates available gaps and programmatically speeds up the audio (up to 1.5x) to prevent dialogue overlaps.
* **Non-Destructive Multiplexing:** Throughout the pipeline, we utilized **FFmpeg** with stream copying (`-c:v copy`). This ensures that while the audio is heavily processed, the video stream is never re-encoded, preserving 100% of the original visual fidelity.

---

##  Challenges & Resolutions
* **The "Frozen Frame" Bug:** We encountered an issue where the video would freeze on the final frame if the new audio was even slightly longer than the video stream. We solved this by implementing **Hard Temporal Anchoring** and the `-shortest` flag during final renders.
* **AI Static & Hiss:** Zero-shot cloning models often generate low-level electronic hiss. We mitigated this by applying **FFT Noise Cancellation** (`afftdn`) with a tuned noise floor of `-25dB` to "de-hiss" the voices during post-production.
* **SRT Syntax Logic:** Converting JSON timestamps into the strict **SubRip (SRT)** format required a custom mathematical utility to handle millisecond precision, ensuring that the French captions align perfectly with the spoken dubbed audio.

---

##  Results & Deliverables
The final output of this notebook is a high-fidelity video file featuring:
1.  **Isolated Backgrounds:** Original music and sound effects are perfectly preserved.
2.  **Cloned French Vocals:** Natural, de-noised, and time-synced dialogue that mimics the original actors.
3.  **Hardcoded Subtitles:** High-legibility French captions (Arial, 22pt, with drop-shadow) burned directly into the frame for maximum accessibility.

# 1) Environment Preparation

### Environment Setup

**Task Objective:** Install the necessary libraries for audio separation and text-to-speech.

**Engineering Choices:** We selected `demucs` for high-quality vocal isolation, `coqui-tts` for natural voice generation, and tools like `pydub` and `noisereduce` for manipulating and cleaning the audio arrays.

In [1]:
!pip install coqui-tts pydub -q
!pip install noisereduce scipy -q
!pip install demucs

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 16.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.3 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 6.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.

### Model Initialization: Voice Cloning Engine

**Task Objective:** Download and load the XTTS-v2 text-to-speech model into the GPU.

**Engineering Choices:** We selected `XTTS-v2` for its advanced zero-shot voice cloning capabilities. Loading the model directly to the GPU (`.to("cuda")`) is crucial for reducing inference time and accelerating the audio generation process.

In [2]:
import torch
from TTS.api import TTS

print("Downloading the XTTS-v2 Voice Cloning Model...")
print("If a box pops up asking you to agree to the terms, type 'y' and press Enter!\n")

# Load the model into the GPU
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")

print("\n✅ XTTS-v2 is loaded and ready to clone voices!")

If a box pops up asking you to agree to the terms, type 'y' and press Enter!

 > You must confirm the following:
 | > "I have purchased a commercial license from Coqui: licensing@coqui.ai"
 | > "Otherwise, I agree to the terms of the non-commercial CPML: https://coqui.ai/cpml" - [y/n]


 | | >  y


100%|██████████| 1.87G/1.87G [00:20<00:00, 91.2MiB/s]
4.37kiB [00:00, 5.59MiB/s]
361kiB [00:00, 29.2MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 56.0kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 59.7MiB/s]



✅ XTTS-v2 is loaded and ready to clone voices!


### Library Imports & Tool Setup

**Task Objective:** Import the core system utilities, deep learning frameworks, and audio manipulation tools required for the pipeline.

**Engineering Choices:** We imported system modules (`os`, `subprocess`, `glob`, `json`) to handle file management, data parsing, and external command execution. For precise audio control, specific `pydub` functions (`speedup`, `detect_nonsilent`) were selected to adjust audio pacing and analyze speech intervals, which is critical for syncing the new dub to the original video timeline. `torch` and `TTS` are included to drive the text-to-speech generation.

In [12]:
import json
import os
import torch
import warnings
import subprocess
import glob
from pydub import AudioSegment
from pydub.effects import speedup  
from pydub.silence import detect_nonsilent 
from TTS.api import TTS


# 2) Video Dubbing

### 1-Pipeline Initialization & Data Paths

This section prepares the notebook for the final synchronization phase. It suppresses non-critical warnings to keep the output clean and establishes the absolute file paths within the Kaggle environment for both the base video and the translated JSON transcript.

In [4]:

warnings.filterwarnings("ignore") 
print("🎬 STARTING THE ADVANCED SYNC PIPELINE...\n")

# --- YOUR KAGGLE FILE PATHS ---
video_path = "/kaggle/input/datasets/djebril/dubdata/final_dubbed_video.mp4"
transcript_path = "/kaggle/input/datasets/djebril/dubdata/translated_transcript_fixed.json" 
# ------------------------------

🎬 STARTING THE ADVANCED SYNC PIPELINE...



### 2-Audio Extraction & Data Ingestion

This step handles the initial data processing. We use `ffmpeg` to strip the original audio track from the video file. The audio is specifically formatted (mono, 22050 Hz) because the XTTS-v2 voice cloning model requires these precise specifications to use the audio as a reference voice. Finally, we load the translated JSON transcript, which provides the text and exact timestamps needed for the new dub.

In [5]:
# 1. Extract base audio
print("1️⃣ Extracting original audio for voice cloning references...")
os.system(f"ffmpeg -i {video_path} -vn -acodec pcm_s16le -ar 22050 -ac 1 base_audio.wav -y -loglevel error")
base_audio = AudioSegment.from_wav("base_audio.wav")

with open(transcript_path, "r", encoding="utf-8") as f:
    data = json.load(f)

1️⃣ Extracting original audio for voice cloning references...


### 3-Automatic Vocal Isolation & Reference Extraction

This section executes a critical two-step process to prepare high-quality source material for the voice cloning engine. 

First, we use `demucs` (accelerated via GPU) to isolate the vocal track from the original audio's background music and sound effects. This is a crucial engineering choice: if we feed raw, unseparated audio into the XTTS-v2 model, the AI will attempt to clone the background noise along with the voice, resulting in distorted outputs. 

Next, the script parses the transcript to automatically extract a reference audio file for each unique speaker. Instead of simply grabbing the first line a speaker says, the algorithm evaluates all of their lines and selects the clip whose duration is closest to **6.0 seconds**. We implemented this specific heuristic because ~6 seconds is the optimal sweet spot for zero-shot cloning—it provides the model with enough phonetic data to accurately capture the speaker's unique tone and cadence, while avoiding overly long clips that might contain extended pauses or strain system memory.

In [6]:
# 2. AUTOMATIC VOCAL ISOLATION & REFERENCE EXTRACTION
print("2️⃣  Fully Automated Vocal Isolation (Removing Music/SFX)...")
subprocess.run(["demucs", "-d", "cuda", "--two-stems", "vocals", "base_audio.wav"], check=True)

vocal_stem_path = "separated/htdemucs/base_audio/vocals.wav"
if not os.path.exists(vocal_stem_path):
    print("⚠️ Path adjustment: Searching for vocal stem...")
    vocal_stem_path = glob.glob("separated/*/base_audio/vocals.wav")[0]

clean_vocals = AudioSegment.from_wav(vocal_stem_path)

refs = {}
for line in data:
    spk = line["speaker"]
    duration = line["end"] - line["start"]
    
    if spk not in refs:
        refs[spk] = {"start": line["start"], "end": line["end"], "duration": duration}
    else:
        current_best_diff = abs(refs[spk]["duration"] - 6.0)
        new_diff = abs(duration - 6.0)
        if new_diff < current_best_diff:
            refs[spk] = {"start": line["start"], "end": line["end"], "duration": duration}

for spk, info in refs.items():
    start_ms = int(info["start"] * 1000)
    end_ms = int(info["end"] * 1000)
    clean_vocals[start_ms:end_ms].export(f"ref_{spk}.wav", format="wav")
    print(f"    -> Extracted music-free reference for {spk} ({info['duration']:.2f}s)")

2️⃣  Fully Automated Vocal Isolation (Removing Music/SFX)...
Important: the default model was recently changed to `htdemucs` the latest Hybrid Transformer Demucs model. In some cases, this model can actually perform worse than previous models. To get back the old default model use `-n mdx_extra_q`.
Downloading: "https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/955717e8-8726e21a.th" to /root/.cache/torch/hub/checkpoints/955717e8-8726e21a.th


100%|██████████| 80.2M/80.2M [00:00<00:00, 159MB/s] 


Selected model is a bag of 1 models. You will see that many progress bars per track.
Separated tracks will be stored in /kaggle/working/separated/htdemucs
Separating track base_audio.wav


100%|██████████████████████████████████████████████| 251.54999999999998/251.54999999999998 [00:10<00:00, 23.11seconds/s]
/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:178: UserWarning: The 'encoding' parameter is not fully supported by TorchCodec AudioEncoder.
  return save_with_torchcodec(
/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:178: UserWarning: The 'bits_per_sample' parameter is not directly supported by TorchCodec AudioEncoder.
  return save_with_torchcodec(


    -> Extracted music-free reference for SPEAKER_00 (5.92s)
    -> Extracted music-free reference for SPEAKER_01 (3.02s)


### 4-Voice Cloning & Dynamic Synchronization

**Task Objective:** Generate the translated French dialogue using the cloned voices and precisely synchronize it to the original video timeline.

**Engineering Choices:** This is the core engine of the dubbing pipeline, and several specific logic choices were made to handle the complexities of audio translation:
* **Strict Temporal Anchoring:** We overlay the generated audio at the exact `actual_start_ms` of the original line. We do not delay start times, ensuring that audio drift doesn't accumulate over the length of the video.
* **Silence Trimming:** AI voice models often generate unpredictable fractions of a second of silence before speaking. We use `detect_nonsilent` to strip this out so the actual spoken words align perfectly with the character's lip movements.
* **The "Gap Squisher" & Overlap Logic:** French dialogue often takes longer to speak than the original language. To fit the new audio into the original visual window without bleeding over other lines, we implemented a dynamic time-stretching algorithm:
    * We calculate the available gap before the next speaker starts.
    * If the *same* character continues speaking, we allow up to a 1000ms overlap (simulating natural speech continuation). If a *different* character interrupts, we restrict the overlap to a strict 200ms.
    * If the generated clip exceeds this safe window, we use `pydub`'s `speedup` function to compress it. A hard safety cap of 1.5x speed is enforced to prevent the audio from sounding artificially pitched or distorted.

In [7]:
# 3. Clone and Auto-Sync
print("\n3️⃣ Cloning voices and Auto-Syncing French dialogue! (This takes a moment...)")
total_duration_ms = int(data[-1]["end"] * 1000) + 15000 
master_french_audio = AudioSegment.silent(duration=total_duration_ms) 

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")

for i, line in enumerate(data):
    spk = line["speaker"]
    french_text = line["french_translation"]
    
    # ==========================================
    # STRICT ANCHORING: NEVER delay the start time!
    # ==========================================
    actual_start_ms = int(line["start"] * 1000)
    target_duration_sec = line["end"] - line["start"]
        
    current_ref = f"ref_{spk}.wav"
    raw_path = f"raw_line_{i}.wav"
    
    print(f"   🎙️ [{spk}]: {french_text}")
    
    estimated_natural_duration = len(french_text) / 14.0
    ai_speed = estimated_natural_duration / target_duration_sec
    ai_speed = max(1.0, min(ai_speed, 1.30)) 
    
    tts.tts_to_file(
        text=french_text, 
        speaker_wav=current_ref, 
        language="fr", 
        file_path=raw_path,
        speed=ai_speed 
    )
    
    temp_audio = AudioSegment.from_wav(raw_path)
    nonsilent_ranges = detect_nonsilent(temp_audio, min_silence_len=50, silence_thresh=-38)
    
    if nonsilent_ranges:
        trim_start = nonsilent_ranges[0][0]
        trim_end = nonsilent_ranges[-1][1]
        trimmed_audio = temp_audio[trim_start:trim_end]
    else:
        trimmed_audio = temp_audio
        
    # ==========================================
    # THE REPAIRED GAP SQUISHER
    # ==========================================
    if i + 1 < len(data):
        next_start_ms = int(data[i+1]["start"] * 1000)
        is_same_speaker = data[i+1]["speaker"] == spk
        
        if is_same_speaker:
            # If the actor is talking to himself, allow up to 1 full second of overlap
            gap_duration_ms = (next_start_ms - actual_start_ms) + 1000
        else:
            # If the OTHER actor interrupts, strict 200ms overlap
            gap_duration_ms = (next_start_ms - actual_start_ms) + 200
            
        # SAFETY FLOOR: Never allow the window to shrink below 500ms
        max_safe_duration_ms = max(500, gap_duration_ms)
    else:
        max_safe_duration_ms = len(trimmed_audio) + 3000

    if len(trimmed_audio) > max_safe_duration_ms:
        speed_ratio = len(trimmed_audio) / max_safe_duration_ms
        # SAFETY CAP: Never compress the audio more than 1.5x speed
        safe_ratio = min(speed_ratio, 1.5)
        
        trimmed_audio = speedup(trimmed_audio, playback_speed=safe_ratio, chunk_size=30, crossfade=15)
        
        if len(trimmed_audio) > max_safe_duration_ms:
            trimmed_audio = trimmed_audio[:max_safe_duration_ms].fade_out(100)
        
        print(f"    🗜️ Squished audio to fit (Ratio: {safe_ratio:.2f}x).")
    else:
        print(f"    ✅ Audio fits naturally.")

    trimmed_audio = trimmed_audio.fade_in(30).fade_out(30)
    
    master_french_audio = master_french_audio.overlay(trimmed_audio, position=actual_start_ms)



3️⃣ Cloning voices and Auto-Syncing French dialogue! (This takes a moment...)
   🎙️ [SPEAKER_00]: Puis-je vous aider, monsieur ?
    ✅ Audio fits naturally.
   🎙️ [SPEAKER_00]: Diet Coke, s'il vous plaît.
    ✅ Audio fits naturally.
   🎙️ [SPEAKER_00]: Et cinq minutes de votre temps.
    ✅ Audio fits naturally.
   🎙️ [SPEAKER_01]: Que puis-je faire pour vous ? 
    ✅ Audio fits naturally.
   🎙️ [SPEAKER_00]: S'il vous plaît, asseyez-vous. 
    ✅ Audio fits naturally.
   🎙️ [SPEAKER_00]: J'aimerais savoir pourquoi vous n'avez pas voulu me rencontrer hier. 
    ✅ Audio fits naturally.
   🎙️ [SPEAKER_00]: Je suis désolée, je ne vous suis pas. 
    ✅ Audio fits naturally.
   🎙️ [SPEAKER_00]: J'étais assise ici hier en attendant de rencontrer quelqu'un. 
    ✅ Audio fits naturally.
   🎙️ [SPEAKER_00]: Je crois que cette personne, c'était toi.
    ✅ Audio fits naturally.
   🎙️ [SPEAKER_01]: Je pense que vous me confondez avec quelqu'un d'autre. 
    ✅ Audio fits naturally.
   🎙️ [SPEAKER_01

### The Freeze-Frame Fix & Final Export

**Task Objective:** Trim the generated master audio track to match the exact length of the original video and export the final file.

**Engineering Choices:** We implemented a hard cutoff (`[:len(base_audio)]`) to solve a common automated dubbing bug: if the generated audio is even a few milliseconds longer than the original video, the final `.mp4` will freeze on the last frame while the audio finishes. Finally, we export the synced track as a lossless `.wav` file to preserve maximum audio quality before it gets multiplexed back together with the video stream.

In [8]:
# ==========================================
# 4. THE FREEZE-FRAME FIX
# ==========================================
master_french_audio = master_french_audio[:len(base_audio)]

output_filename = "final_french_dub_SYNCED.wav"
master_french_audio.export(output_filename, format="wav")
print(f"\n✅ SYNC COMPLETE! The fully synced audio is saved as {output_filename}")


✅ SYNC COMPLETE! The fully synced audio is saved as final_french_dub_SYNCED.wav


### 5-Final Video Assembly & Multiplexing

**Task Objective:** Combine the original video stream with the newly generated, synchronized French audio track into a final `.mp4` file.

**Engineering Choices:** We utilized `ffmpeg` for the final multiplexing phase for its speed and precision. By using the `-c:v copy` argument, we pass the video stream through without re-encoding it—this preserves 100% of the original visual quality and takes seconds rather than minutes. The `-map 0:v:0 -map 1:a:0` flags explicitly tell the system to take the video from the first input and the audio from our newly generated dub, cleanly stripping the original sound. Finally, adding the `-shortest` flag acts as a definitive fail-safe to ensure the output ends exactly when the audio does, completely eliminating any chance of a frozen frame at the end of the video.

In [11]:
# ==========================================
# 5. FINAL VIDEO MERGE
# ==========================================
print("\n🎞️ Merging final audio with original video...")
final_video_output = "final_french_dub_SYNCED.mp4"

merge_command = (
    f"ffmpeg -i {video_path} -i {output_filename} "
    f"-c:v copy -map 0:v:0 -map 1:a:0 -shortest {final_video_output} -y -loglevel error"
)

os.system(merge_command)
print(f"🎉 DONE! Your fully dubbed video without frozen frames is saved as: {final_video_output}")


🎞️ Merging final audio with original video...
🎉 DONE! Your fully dubbed video without frozen frames is saved as: final_french_dub_SYNCED.mp4


# 3) Audio Mixing

### 1-Post-Production Initialization

This section sets up the final phase of the pipeline: audio cleanup. We define the input file path for the newly synchronized French dub, establish the output destination for the noise-reduced version, and reference the base video file to prepare for the final stages of audio processing.

In [17]:
print("🎧 STARTING POST-PRODUCTION & NOISE CANCELLATION...\n")

# --- FILE PATHS ---
french_dub_path = "final_french_dub_SYNCED.wav"
cleaned_dub_path = "final_french_dub_CLEANED.wav"
video_path = "/kaggle/input/datasets/djebril/dubdata/final_dubbed_video.mp4" 

🎧 STARTING POST-PRODUCTION & NOISE CANCELLATION...



### 2-Studio Noise Cancellation (De-Hiss)

**Task Objective:** Remove the low-level background static and hiss generated during the AI voice cloning process.

**Engineering Choices:** We implemented FFmpeg's Fast Fourier Transform DeNoise (`afftdn`) filter directly via the command line for maximum processing speed. AI text-to-speech models, particularly zero-shot cloners like XTTS, often introduce subtle artifacts or baseline static into the audio. By specifically tuning the noise floor parameter to `nf=-25`, we aggressively target this artificial hiss without degrading the natural frequencies or muddying the clarity of the spoken French dialogue.

In [18]:
# ==========================================
# 1. STUDIO NOISE CANCELLATION (De-Hiss)
# ==========================================
print("🧹 1. Applying FFT Noise Cancellation to the French voices...")
# We use FFmpeg's 'afftdn' (Audio Fast Fourier Transform DeNoise) filter.
# nf=-25 targets the low-level static/hiss that XTTS sometimes generates.
denoise_cmd = f'ffmpeg -i {french_dub_path} -af "afftdn=nf=-25" {cleaned_dub_path} -y -loglevel error'
os.system(denoise_cmd)
print("   ✅ AI voices cleaned and static removed.")

🧹 1. Applying FFT Noise Cancellation to the French voices...
   ✅ AI voices cleaned and static removed.


### 3-Background Audio Retrieval (Music & SFX)

**Task Objective:** Locate and prepare the isolated background audio track that contains the original music and sound effects.

**Engineering Choices:** We utilized a `try-except` block paired with `glob` for dynamic path resolution. Because the `demucs` library can generate slightly different output folder structures depending on the specific model used (e.g., `htdemucs`), hardcoding the absolute path is fragile and prone to breaking. Using `glob` allows the script to reliably find `no_vocals.wav` regardless of the exact subfolder name. The exception handling ensures that if the separation step failed earlier, the pipeline will halt immediately with a clear, human-readable error rather than failing silently later in the merge phase.

In [19]:
# ==========================================
# 2. LOCATE BACKGROUND AUDIO
# ==========================================
try:
    bg_stem_path = glob.glob("separated/*/base_audio/no_vocals.wav")[0]
    print(f"\n✅ 2. Found original background audio at: {bg_stem_path}")
except IndexError:
    print("\n❌ ERROR: Could not find no_vocals.wav. Make sure Demucs ran properly!")
    raise


✅ 2. Found original background audio at: separated/htdemucs/base_audio/no_vocals.wav


### 4-The Post-Production Master Mix

**Task Objective:** Combine the de-noised French voice track with the original background audio to create the final, studio-ready master soundscape.

**Engineering Choices:** To ensure the new AI-generated dialogue is clearly intelligible, we implemented an audio engineering technique called "ducking." By programmatically reducing the volume of the background track by 6 decibels (`bg_audio - 6.0`), we carve out acoustic space for the French voices so they aren't drowned out by the original music or sound effects. After overlaying the tracks, we once again apply a hard temporal cutoff (`[:len(bg_audio)]`) as a final fail-safe. This guarantees the master mix matches the exact video duration, preventing any potential sync drift or frozen frames in the final render.

In [20]:

# ==========================================
# 3. THE POST-PRODUCTION MIX
# ==========================================
print("\n📥 3. Loading audio tracks into the mixer...")
bg_audio = AudioSegment.from_wav(bg_stem_path)
french_vocals_clean = AudioSegment.from_wav(cleaned_dub_path)

print("🎛️    Ducking background volume (-6dB) and layering clean voices...")
bg_audio = bg_audio - 6.0 

# Overlay the CLEANED French voices on top of the background music/noise
final_mix = bg_audio.overlay(french_vocals_clean)

# Ensure the final mix is exactly the same length as the video
final_mix = final_mix[:len(bg_audio)]

# Save the master audio track
mixed_audio_output = "MASTER_MIX_AUDIO.wav"
final_mix.export(mixed_audio_output, format="wav")
print(f"   ✅ Master mix audio saved as {mixed_audio_output}")



📥 3. Loading audio tracks into the mixer...
🎛️    Ducking background volume (-6dB) and layering clean voices...
   ✅ Master mix audio saved as MASTER_MIX_AUDIO.wav


### 5-Final Cinematic Video Multiplexing

**Task Objective:** Combine the fully mixed, studio-quality audio track with the original video stream to produce the final dubbed output.

**Engineering Choices:** We rely on `ffmpeg` for this final assembly step. By using `-c:v copy`, we pass the original video stream through without re-encoding, which preserves 100% of the visual fidelity and completes the process in seconds rather than minutes. The explicit mapping flags (`-map 0:v:0 -map 1:a:0`) cleanly strip out the old audio and replace it entirely with our new master mix. Finally, the `-shortest` flag is applied as a strict guardrail to guarantee the video ends exactly with the audio, preventing any trailing frozen frames or dead air.

In [21]:
# ==========================================
# 4. FINAL VIDEO MULTIPLEXING
# ==========================================
print("\n🎞️ 4. Merging the Master Mix with the original video...")
final_cinema_output = "THE_ULTIMATE_CINEMATIC_DUB.mp4"

merge_command = (
    f"ffmpeg -i {video_path} -i {mixed_audio_output} "
    f"-c:v copy -map 0:v:0 -map 1:a:0 -shortest {final_cinema_output} -y -loglevel error"
)

os.system(merge_command)
print(f"🎉 BOOM! Post-production complete.")
print(f"🎬 Your studio-quality movie (De-noised voices + Background effects) is saved as: {final_cinema_output}")


🎞️ 4. Merging the Master Mix with the original video...
🎉 BOOM! Post-production complete.
🎬 Your studio-quality movie (De-noised voices + Background effects) is saved as: THE_ULTIMATE_CINEMATIC_DUB.mp4


# 4)Subtitle Generation

### 1-Subtitle Generation Initialization

This section prepares the workspace for the final major feature of the project: generating and burning French subtitles into the video. It defines the necessary file paths, crucially taking the fully mixed, studio-quality video from our previous post-production step (`THE_ULTIMATE_CINEMATIC_DUB.mp4`) and using it as the new base input. It also establishes the output path for the intermediary SubRip subtitle file (`.srt`) that will be generated from our JSON transcript.

In [22]:

print("📝 STARTING SUBTITLE GENERATION...\n")

# --- FILE PATHS ---
# We are using the output from your previous post-production cell!
transcript_path = "/kaggle/input/datasets/djebril/dubdata/translated_transcript_fixed.json" 
input_video = "THE_ULTIMATE_CINEMATIC_DUB.mp4"
output_video = "THE_ULTIMATE_CINEMATIC_DUB_WITH_SUBS.mp4"
srt_path = "french_subtitles.srt"


📝 STARTING SUBTITLE GENERATION...



### 2-Subtitle Generation: JSON to SRT Conversion

**Task Objective:** Parse the translated JSON transcript and convert it into a standardized SubRip Subtitle (`.srt`) file.

**Engineering Choices:** * **Strict Time Formatting:** SRT files require a highly specific timecode syntax (`HH:MM:SS,mmm`). We implemented a custom `format_srt_time` helper function to mathematically convert raw floating-point seconds from our JSON into this exact string format, ensuring compatibility with standard video players without needing to import heavy external datetime libraries.
* **Text Sanitization:** AI-generated text can sometimes include stray markdown asterisks or quotation marks. We implemented `.replace()` methods to actively strip these artifacts out (`.replace('"', '').replace('*', '')`), ensuring the final subtitles look professional and display cleanly without rendering glitches.
* **Encoding Guardrails:** Explicitly forcing `encoding="utf-8"` during both the read and write operations is a crucial choice. Because we are dealing with French translations, this guarantees that all special characters and accents (like é, à, or ç) are perfectly preserved and not corrupted into illegible symbols during file creation.

In [23]:
# ==========================================
# 1. CONVERT JSON TO SRT FORMAT
# ==========================================
print("1️⃣ Converting translation JSON into standard SRT subtitle format...")

def format_srt_time(seconds):
    """Converts raw seconds into the strict HH:MM:SS,mmm format required by SRT files."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millisecs = int((seconds - int(seconds)) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millisecs:03d}"

with open(transcript_path, "r", encoding="utf-8") as f:
    data = json.load(f)

with open(srt_path, "w", encoding="utf-8") as f:
    for i, line in enumerate(data):
        # SRT block index starts at 1
        f.write(f"{i + 1}\n")
        
        # Write the time boundaries
        start_time = format_srt_time(line["start"])
        end_time = format_srt_time(line["end"])
        f.write(f"{start_time} --> {end_time}\n")
        
        # Clean the text and write the dialogue
        text = line["french_translation"].replace('"', '').replace('*', '').strip()
        f.write(f"{text}\n\n")

print(f"   ✅ Subtitles successfully built and saved to {srt_path}")


1️⃣ Converting translation JSON into standard SRT subtitle format...
   ✅ Subtitles successfully built and saved to french_subtitles.srt


### 3-Final Cinematic Render: Hardcoded Subtitles

**Task Objective:** Permanently embed (burn) the French subtitles into the video frames to ensure they are visible on any device or platform without requiring external files.

**Engineering Choices:** * **Cinematic Styling:** We utilized FFmpeg’s `force_style` parameter to implement high-legibility "broadcast" styling. By setting a white primary color (`&H00FFFFFF`) with a black outline (`Outline=2`) and a drop shadow (`Shadow=1`), we ensure the subtitles remain readable regardless of the background imagery (e.g., bright scenes vs. dark scenes).
* **Audio Stream Preservation:** A critical choice was made to use `-c:a copy`. Since we already spent significant time cleaning, de-hissing, and mixing the audio in previous steps, "copying" the stream prevents any further lossy compression, maintaining the studio-quality master mix.
* **Vertical Positioning:** The `MarginV=25` parameter was selected to lift the text slightly off the very bottom of the frame, preventing the subtitles from being obscured by UI elements like progress bars or playback controls on video hosting platforms.
* **Error Handling:** Wrapping the execution in a `subprocess.run` with `check=True` provides a safety net; if the subtitle file is missing or the video stream is corrupted, the script identifies the failure immediately rather than creating a broken file.

In [24]:

# ==========================================
# 2. BURN SUBTITLES INTO VIDEO (Hardcoding)
# ==========================================
print("\n🔥 2. Burning subtitles permanently into the video frames...")

# THE CINEMATIC STYLE: Arial Font, Size 22, White Text, Black Outline, Drop Shadow.
style = "FontName=Arial,FontSize=22,PrimaryColour=&H00FFFFFF,OutlineColour=&H00000000,BorderStyle=1,Outline=2,Shadow=1,MarginV=25"

# Note: We use -c:a copy. This tells FFmpeg to copy your perfect Master Audio Mix 
# exactly as it is without touching or compressing the sound quality!
burn_cmd = (
    f'ffmpeg -i {input_video} '
    f'-vf "subtitles={srt_path}:force_style=\'{style}\'" '
    f'-c:a copy {output_video} -y -loglevel error'
)

# Run the command
try:
    subprocess.run(burn_cmd, shell=True, check=True)
    print("\n🎉 BOOM! Your final masterpiece is officially complete.")
    print(f"🎬 Video with hardcoded French subtitles saved as: {output_video}")
except subprocess.CalledProcessError:
    print("\n❌ ERROR: FFmpeg failed to burn the subtitles.")
    print(f"Please make sure '{input_video}' actually exists in your working folder!")


🔥 2. Burning subtitles permanently into the video frames...

🎉 BOOM! Your final masterpiece is officially complete.
🎬 Video with hardcoded French subtitles saved as: THE_ULTIMATE_CINEMATIC_DUB_WITH_SUBS.mp4
